In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
train = pd.read_json('/content/drive/MyDrive/DL-TH4/data/train.json')
dev = pd.read_json('/content/drive/MyDrive/DL-TH4/data/dev.json')
test = pd.read_json('/content/drive/MyDrive/DL-TH4/data/test.json')

In [3]:
train = train.sample(n=20000, random_state=42)
dev = dev.sample(n=2000, random_state=42)
test = test.sample(n=2000, random_state=42)

In [4]:
%%writefile /content/drive/MyDrive/DL-TH4/Vocab.py
import os
import json
import torch
import re

class Vocab:
    def __init__(self, src_language: str, tgt_language: str):
        self.src_language = src_language
        self.tgt_language = tgt_language
        self.initialize_special_tokens()

    def initialize_special_tokens(self):
        self.pad_token = "<pad>"
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"
        self.specials = [self.pad_token, self.bos_token, self.eos_token, self.unk_token]
        self.pad_idx = 0
        self.bos_idx = 1
        self.eos_idx = 2
        self.unk_idx = 3

    def preprocess_sentence(self, sentence: str):
        sentence = sentence.lower()
        return re.findall(r"\w+", sentence)

    def make_vocab(self, path: str):
        src_words = set()
        tgt_words = set()
        for file in os.listdir(path):
            data = json.load(open(os.path.join(path, file), encoding="utf-8"))
            for item in data:
                src_words.update(self.preprocess_sentence(item[self.src_language]))
                tgt_words.update(self.preprocess_sentence(item[self.tgt_language]))
        src_itos = self.specials + list(src_words)
        tgt_itos = self.specials + list(tgt_words)
        self.src_itos = {i: tok for i, tok in enumerate(src_itos)}
        self.src_stoi = {tok: i for i, tok in enumerate(src_itos)}
        self.tgt_itos = {i: tok for i, tok in enumerate(tgt_itos)}
        self.tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}

    def encode_sentence(self, sentence: str, language: str):
        tokens = self.preprocess_sentence(sentence)
        stoi = self.src_stoi if language == self.src_language else self.tgt_stoi
        vec = [self.bos_idx] + [stoi.get(tok, self.unk_idx) for tok in tokens] + [self.eos_idx]
        return torch.tensor(vec, dtype=torch.long)

    def decode_sentence(self, vec: torch.Tensor, language: str):
        ids = vec.tolist()
        itos = self.src_itos if language == self.src_language else self.tgt_itos
        words = []
        for idx in ids:
            if idx == self.eos_idx:
                break
            words.append(itos[idx])
        return " ".join(words)

    def total_src_tokens(self):
        return len(self.src_itos)

    def total_tgt_tokens(self):
        return len(self.tgt_itos)

Overwriting /content/drive/MyDrive/DL-TH4/Vocab.py


In [5]:
%%writefile /content/drive/MyDrive/DL-TH4/Dataset.py
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
import json

class PhoMTDataset(Dataset):
    def __init__(self, path, src_vocab, tgt_vocab):
        with open(path, encoding="utf-8") as f:
            self.data = json.load(f)
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        # Encode sẵn để DataLoader nhanh
        self.data = [{"en": src_vocab.encode_sentence(item["english"], "english"),
                      "vi": tgt_vocab.encode_sentence(item["vietnamese"], "vietnamese")}
                     for item in self.data]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    en = [item["en"] for item in batch]
    vi = [item["vi"] for item in batch]
    en = pad_sequence(en, batch_first=True, padding_value=0)
    vi = pad_sequence(vi, batch_first=True, padding_value=0)
    return en, vi

Overwriting /content/drive/MyDrive/DL-TH4/Dataset.py


# Bài 1

In [6]:
%%writefile /content/drive/MyDrive/DL-TH4/LSTM.py
import torch
import torch.nn as nn

class Seq2SeqLSTM(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, pad_idx):
        super().__init__()
        self.hidden_size = 256
        self.num_layers = 3
        self.src_embedding = nn.Embedding(src_vocab_size, 256, padding_idx=pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, 256, padding_idx=pad_idx)
        self.encoder = nn.LSTM(256, 256, 3, batch_first=True)
        self.decoder = nn.LSTM(256, 256, 3, batch_first=True)
        self.fc = nn.Linear(256, tgt_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx)

    def forward(self, src, tgt):
        embedded_src = self.src_embedding(src)
        _, (h, c) = self.encoder(embedded_src)
        embedded_tgt = self.tgt_embedding(tgt[:, :-1])
        outputs, _ = self.decoder(embedded_tgt, (h, c))
        logits = self.fc(outputs)
        return logits

    def compute_loss(self, logits, tgt):
        return self.loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))


Overwriting /content/drive/MyDrive/DL-TH4/LSTM.py


In [9]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6d7c1db59e81f7ffe36eac4266d7ea22c2c557b4eaed008f982738d00c878897
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [7]:
%%writefile /content/drive/MyDrive/DL-TH4/train.py
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from LSTM import Seq2SeqLSTM
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Vocab
src_vocab = Vocab("english", "vietnamese")
tgt_vocab = Vocab("english", "vietnamese")
data_path = '/content/drive/MyDrive/DL-TH4/data/'
src_vocab.make_vocab(data_path)
tgt_vocab.make_vocab(data_path)

# Dataset
train_dataset = PhoMTDataset(data_path + 'train.json', src_vocab, tgt_vocab)
dev_dataset   = PhoMTDataset(data_path + 'dev.json', src_vocab, tgt_vocab)

# Sample nhỏ
train_dataset.data = train_dataset.data[:2000]
dev_dataset.data   = dev_dataset.data[:500]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# Model
model = Seq2SeqLSTM(src_vocab.total_src_tokens(), tgt_vocab.total_tgt_tokens(), src_vocab.pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train 3 epoch
num_epochs = 3
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for en_batch, vi_batch in tqdm(train_loader):
        en_batch, vi_batch = en_batch.to(device), vi_batch.to(device)
        optimizer.zero_grad()
        logits = model(en_batch, vi_batch)
        loss = model.compute_loss(logits, vi_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{num_epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")

# Save model test
torch.save(model.state_dict(), '/content/drive/MyDrive/DL-TH4/seq2seq_test.pth')


Overwriting /content/drive/MyDrive/DL-TH4/train_eval.py


In [11]:
!python /content/drive/MyDrive/DL-TH4/train_eval.py

100% 125/125 [00:28<00:00,  4.33it/s]
Epoch 1/3, Avg Loss: 7.3058
100% 125/125 [00:28<00:00,  4.33it/s]
Epoch 2/3, Avg Loss: 6.3710
100% 125/125 [00:30<00:00,  4.15it/s]
Epoch 3/3, Avg Loss: 6.3219
ROUGE-L trung bình: 0.155650774943875


# Bài 2

In [14]:
%%writefile "/content/drive/MyDrive/DL-TH4/LSTM_Attention.py"
import torch
import torch.nn as nn
import torch.nn.functional as F

class Seq2SeqAttention(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, pad_idx, hidden_size=256, num_layers=3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Embedding
        self.src_embedding = nn.Embedding(src_vocab_size, hidden_size, padding_idx=pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, hidden_size, padding_idx=pad_idx)

        # Encoder & Decoder
        self.encoder = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)

        # Attention
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

        # Output
        self.fc = nn.Linear(hidden_size, tgt_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx)

    def forward(self, src, tgt):
        # src: (B, S), tgt: (B, T)
        embedded_src = self.src_embedding(src)
        enc_outputs, (h, c) = self.encoder(embedded_src)

        embedded_tgt = self.tgt_embedding(tgt[:, :-1])
        batch_size, tgt_len, _ = embedded_tgt.size()
        dec_outputs = []

        dec_h, dec_c = h, c
        for t in range(tgt_len):
            dec_input = embedded_tgt[:, t:t+1, :]
            dec_output, (dec_h, dec_c) = self.decoder(dec_input, (dec_h, dec_c))

            # Attention
            attn_weights = self._attention(enc_outputs, dec_output)
            context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs)
            dec_output = dec_output + context

            dec_outputs.append(dec_output)

        outputs = torch.cat(dec_outputs, dim=1)
        logits = self.fc(outputs)
        return logits

    def _attention(self, enc_outputs, dec_hidden):
        seq_len = enc_outputs.size(1)
        dec_hidden_exp = dec_hidden.expand(-1, seq_len, -1)
        energy = torch.tanh(self.attn(torch.cat((dec_hidden_exp, enc_outputs), dim=2)))
        attn_scores = self.v(energy).squeeze(2)
        return F.softmax(attn_scores, dim=1)

    def compute_loss(self, logits, tgt):
        return self.loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))


Writing /content/drive/MyDrive/DL-TH4/LSTM_Attention.py


In [15]:
%%writefile /content/drive/MyDrive/DL-TH4/train_eval_attention.py
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from LSTM_Attention import Seq2SeqAttention
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Vocab
data_path = '/content/drive/MyDrive/DL-TH4/data/'
src_vocab = Vocab("english", "vietnamese")
tgt_vocab = Vocab("english", "vietnamese")
src_vocab.make_vocab(data_path)
tgt_vocab.make_vocab(data_path)

train_dataset = PhoMTDataset(data_path + 'train.json', src_vocab, tgt_vocab)
dev_dataset   = PhoMTDataset(data_path + 'dev.json', src_vocab, tgt_vocab)
train_dataset.data = train_dataset.data[:2000]
dev_dataset.data   = dev_dataset.data[:500]

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn, num_workers=0)
dev_loader   = DataLoader(dev_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn, num_workers=0)

# Model
model = Seq2SeqAttention(src_vocab.total_src_tokens(), tgt_vocab.total_tgt_tokens(), src_vocab.pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train 3 epoch
num_epochs = 3
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for en_batch, vi_batch in tqdm(train_loader):
        en_batch, vi_batch = en_batch.to(device), vi_batch.to(device)
        optimizer.zero_grad()
        logits = model(en_batch, vi_batch)
        loss = model.compute_loss(logits, vi_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{num_epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")

# Save model
torch.save(model.state_dict(), '/content/drive/MyDrive/DL-TH4/seq2seq_attention_sample.pth')

# Eval ROUGE-L
model.eval()
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = []
for en_batch, vi_batch in dev_loader:
    en_batch, vi_batch = en_batch.to(device), vi_batch.to(device)
    with torch.no_grad():
        logits = model(en_batch, vi_batch)
        pred_ids = torch.argmax(logits, dim=-1)
    for pred, ref in zip(pred_ids, vi_batch):
        pred_sent = tgt_vocab.decode_sentence(pred, "vietnamese")
        ref_sent  = tgt_vocab.decode_sentence(ref, "vietnamese")
        scores.append(scorer.score(ref_sent, pred_sent)['rougeL'].fmeasure)
print("ROUGE-L trung bình:", sum(scores)/len(scores))

Overwriting /content/drive/MyDrive/DL-TH4/train_eval_attention.py


In [16]:
!python /content/drive/MyDrive/DL-TH4/train_eval_attention.py

100% 125/125 [00:39<00:00,  3.18it/s]
Epoch 1/3, Avg Loss: 7.1414
100% 125/125 [00:38<00:00,  3.22it/s]
Epoch 2/3, Avg Loss: 6.2792
100% 125/125 [00:40<00:00,  3.06it/s]
Epoch 3/3, Avg Loss: 6.1809
ROUGE-L trung bình: 0.14782544290685373


# Bài 3

In [ ]:
%%writefile /content/drive/MyDrive/DL-TH4/LSTM_Attention2.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class Seq2SeqAttention2(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, pad_idx):
        super().__init__()
        self.hidden_size = 256
        self.num_layers = 3

        self.src_embedding = nn.Embedding(src_vocab_size, 256, padding_idx=pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, 256, padding_idx=pad_idx)

        self.encoder = nn.LSTM(256, 256, num_layers=3, batch_first=True)

        self.decoder = nn.LSTM(256, 256, num_layers=3, batch_first=True)

        # Attention
        self.attn = nn.Linear(256 + 256, 256)
        self.v = nn.Linear(256, 1, bias=False)

        self.fc = nn.Linear(256, tgt_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_idx)

    def forward(self, src, tgt):
        embedded_src = self.src_embedding(src)
        encoder_outputs, (h, c) = self.encoder(embedded_src)

        embedded_tgt = self.tgt_embedding(tgt[:, :-1])
        batch_size, tgt_len, _ = embedded_tgt.size()
        outputs = []

        hidden, cell = h, c

        for t in range(tgt_len):
            y_t = embedded_tgt[:, t].unsqueeze(1) 

            # compute attention weights
            repeat_hidden = hidden[-1].unsqueeze(1).repeat(1, encoder_outputs.size(1), 1)
            energy = torch.tanh(self.attn(torch.cat((repeat_hidden, encoder_outputs), dim=2)))
            attn_weights = F.softmax(self.v(energy).squeeze(2), dim=1)  
            context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs) 

            # LSTM input: y_t + context
            lstm_input = y_t + context
            output, (hidden, cell) = self.decoder(lstm_input, (hidden, cell))
            outputs.append(output)

        outputs = torch.cat(outputs, dim=1)  
        logits = self.fc(outputs) 
        return logits

    def compute_loss(self, logits, tgt):
        return self.loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))

Writing /content/drive/MyDrive/DL-TH4/LSTM_Attention2.py


In [ ]:
%%writefile /content/drive/MyDrive/DL-TH4/train_eval_attention2.py
import torch
from torch.utils.data import DataLoader
from Dataset import PhoMTDataset, collate_fn
from Vocab import Vocab
from LSTM_Attention2 import Seq2SeqAttention2  
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Vocab
data_path = '/content/drive/MyDrive/DL-TH4/data/'
src_vocab = Vocab("english", "vietnamese")
tgt_vocab = Vocab("english", "vietnamese")
src_vocab.make_vocab(data_path)
tgt_vocab.make_vocab(data_path)

train_dataset = PhoMTDataset(data_path + 'train.json', src_vocab, tgt_vocab)
dev_dataset   = PhoMTDataset(data_path + 'dev.json', src_vocab, tgt_vocab)
train_dataset.data = train_dataset.data[:2000]
dev_dataset.data   = dev_dataset.data[:500]

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=0)
dev_loader   = DataLoader(dev_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

# Model
model = Seq2SeqAttention2(src_vocab.total_src_tokens(), tgt_vocab.total_tgt_tokens(), src_vocab.pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train
num_epochs = 3
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for en_batch, vi_batch in tqdm(train_loader):
        en_batch, vi_batch = en_batch.to(device), vi_batch.to(device)
        optimizer.zero_grad()
        logits = model(en_batch, vi_batch)
        loss = model.compute_loss(logits, vi_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/{num_epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")

# Save model
torch.save(model.state_dict(), '/content/drive/MyDrive/DL-TH4/seq2seq_attention2_sample.pth')

# Eval ROUGE-L
model.eval()
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = []
for en_batch, vi_batch in dev_loader:
    en_batch, vi_batch = en_batch.to(device), vi_batch.to(device)
    with torch.no_grad():
        logits = model(en_batch, vi_batch)
        pred_ids = torch.argmax(logits, dim=-1)
    for pred, ref in zip(pred_ids, vi_batch):
        pred_sent = tgt_vocab.decode_sentence(pred, "vietnamese")
        ref_sent  = tgt_vocab.decode_sentence(ref, "vietnamese")
        scores.append(scorer.score(ref_sent, pred_sent)['rougeL'].fmeasure)
print("ROUGE-L trung bình:", sum(scores)/len(scores))

Overwriting /content/drive/MyDrive/DL-TH4/train_eval_attention2.py


In [22]:
!python /content/drive/MyDrive/DL-TH4/train_eval_attention2.py

100% 500/500 [01:33<00:00,  5.35it/s]
Epoch 1/3, Avg Loss: 6.8644
100% 500/500 [01:32<00:00,  5.40it/s]
Epoch 2/3, Avg Loss: 6.1990
100% 500/500 [01:32<00:00,  5.43it/s]
Epoch 3/3, Avg Loss: 5.8928
ROUGE-L trung bình: 0.2293132077087271
